# NB1 — Treatment extraction at full scale
### The Peltzman Protocol · MSR 2027

No cloning. The MSR 2024 workflow-history corpus already contains every historical version of every workflow file for ~53,000 repositories, so this notebook reads a dataset instead of crawling GitHub.

**Output:** for every repository, the first month it acquired each safeguard — plus the two treatment definitions the design needs.

| | Treatment | Definition |
|---|---|---|
| **A** | First gating | The month the project first acquires *any* of T1–T4, T6. One collapsed event. |
| **B** | Marginal safeguard | Safeguard T arriving at least three months after the project's first gate. |

---
### This notebook is discovery-first, on purpose

The dataset's paper documents the CSV columns but **not** the internal layout of `workflows.tar.gz`, and the Zenodo page and the paper disagree about column names (`change_type` vs `git_change_type`). Rather than guess, section 2 inspects the real archive and prints what is actually there. Section 3 is a single adapter cell holding every assumption in one place — if the reconnaissance shows something different, that is the only cell you edit.

The join is made robust by construction: instead of trusting the filenames inside the archive, we compute the SHA-256 of each file's bytes ourselves, which is what the dataset uses as `file_hash`. That works whatever the layout turns out to be.


## 0 · Setup

In [ ]:
#@title Configuration { display-mode: "form" }
ZENODO_RECORD = "20340547"   #@param {type:"string"}
WINDOW_START  = "2019-07"    #@param {type:"string"}
WINDOW_END    = "2025-06"    #@param {type:"string"}
MIN_AGE_MONTHS= 12           #@param {type:"integer"}
MARGINAL_GAP  = 3            #@param {type:"integer"}
N_BUCKETS     = 64           #@param {type:"integer"}

import os, sys, json, time, re, shutil, subprocess, hashlib, gzip, tarfile
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError: pass

DRIVE_OK, DRIVE_ERR = False, None
if IN_COLAB:
    try: drive.mount("/content/drive", force_remount=False); DRIVE_OK = True
    except Exception as e:
        DRIVE_ERR = f"{type(e).__name__}: {e}"; print("!! Drive mount FAILED:", DRIVE_ERR)

DRIVE = ("/content/drive/MyDrive/peltzman" if DRIVE_OK else
         "/content/peltzman_LOCAL" if IN_COLAB else os.path.expanduser("~/peltzman_out"))
if IN_COLAB and not DRIVE_OK:
    print("!! No Drive — state on the runtime disk, destroyed on reset. Fix before section 4.")

STATE = Path(DRIVE, "state"); RAW = Path(DRIVE, "raw")     # the 3.9 GB corpus lives here
WORK  = Path("/content/work" if IN_COLAB else "/tmp/work") # extraction target, disposable
CODE  = Path("/content" if IN_COLAB else "/tmp")
for _p in (STATE, RAW, WORK, CODE): Path(_p).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))
print("state:", STATE, "\nraw  :", RAW, "\nwork :", WORK)

In [ ]:
%%writefile /content/peltzman_io.py
"""
Peltzman Protocol - durable, resumable job state.

Guarantee
---------
Kill the process, lose the runtime, close the browser, at ANY instant:
  * no file is ever left half-written,
  * no completed work is ever lost,
  * no work is ever silently duplicated,
  * restarting the cell resumes from the last completed batch.

How
---
The key list is fixed and ordered at job creation, and batch `i` is defined as
a deterministic slice of it. A batch's output is therefore a pure function of
its index, so re-running a batch overwrites its own file and can never
duplicate rows. Completion is represented by the *existence* of that file and
nothing else - there is no separate done-list that can drift out of sync with
the data.

Every file is written to a temporary name, flushed, fsync'd, and then moved
into place with os.replace(), which is atomic. A crash mid-write leaves a
stray .tmp file (harmless, cleaned on next run) and the previous state intact.

Because completion is file existence, the crash window is exactly one batch.
Size batches so one takes a few minutes.
"""
from __future__ import annotations
import json, os, time, hashlib, tempfile, shutil, glob, sys
from pathlib import Path

__all__ = ["Job", "atomic_write_text", "atomic_write_bytes"]


# --------------------------------------------------------------- atomic io --
def atomic_write_bytes(path, data: bytes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())          # no-op on some mounts; harmless
        os.replace(tmp, path)             # atomic within a filesystem
    except BaseException:
        try: os.unlink(tmp)
        except OSError: pass
        raise


def atomic_write_text(path, text: str):
    atomic_write_bytes(path, text.encode("utf-8"))


def _hash_params(params) -> str:
    blob = json.dumps(params, sort_keys=True, default=str).encode()
    return hashlib.sha1(blob).hexdigest()[:10]


# --------------------------------------------------------------------- job --
class Job:
    """A resumable pass over an ordered list of keys.

    >>> job = Job("nb1_workflows", keys=repos, root=DRIVE, batch_size=25,
    ...           params={"detector": "v3"})
    >>> for i, keys in job.pending():
    ...     job.commit(i, [row for k in keys for row in scan(k)])
    >>> df = job.frame()

    Changing `params` starts a new, separate run directory: results computed
    under different settings are never silently mixed.
    """

    def __init__(self, name, keys, root, batch_size=25, params=None, verbose=True):
        self.name = name
        self.params = dict(params or {})
        self.batch_size = int(batch_size)
        self.verbose = verbose
        # NOTE: the key count is deliberately NOT part of the run identity.
        # If the input list changes at all we want a loud error, not a silent
        # fresh run that looks like "nothing was ever done".
        self.dir = Path(root) / name / _hash_params(
            {"params": self.params, "batch_size": self.batch_size})
        self.shards = self.dir / "shards"
        self.shards.mkdir(parents=True, exist_ok=True)

        keyfile = self.dir / "keys.json"
        if keyfile.exists():
            stored = json.loads(keyfile.read_text())
            if stored != list(keys):
                raise ValueError(
                    f"{self.name}: key list changed since this run started "
                    f"({len(stored)} -> {len(keys)}). Change `params` to start "
                    f"a new run, or restore the original list.")
        else:
            atomic_write_text(keyfile, json.dumps(list(keys)))
            atomic_write_text(self.dir / "meta.json", json.dumps({
                "name": name, "created": time.time(), "n_keys": len(keys),
                "batch_size": self.batch_size, "params": self.params,
                "python": sys.version.split()[0]}, indent=1))

        self.keys = list(keys)
        self._sweep_tmp()
        if verbose:
            d, n = self.n_done_batches(), self.n_batches()
            print(f"[{name}] {self.dir}")
            print(f"[{name}] {d}/{n} batches complete"
                  + (f" — resuming at batch {d}" if 0 < d < n else
                     " — nothing to do" if d == n else " — starting fresh"))

    # ---------------------------------------------------------- internals --
    def _sweep_tmp(self):
        """Remove leftovers from a crash mid-write. They are never valid."""
        for t in glob.glob(str(self.shards / "*.tmp")) + glob.glob(str(self.dir / "*.tmp")):
            try: os.unlink(t)
            except OSError: pass

    def _path(self, i) -> Path:
        return self.shards / f"batch-{i:06d}.jsonl"

    def n_batches(self) -> int:
        return (len(self.keys) + self.batch_size - 1) // self.batch_size

    def batch_keys(self, i):
        return self.keys[i * self.batch_size:(i + 1) * self.batch_size]

    def done_batches(self):
        return {i for i in range(self.n_batches()) if self._path(i).exists()}

    def n_done_batches(self) -> int:
        return len(self.done_batches())

    # ------------------------------------------------------------- driving --
    def pending(self):
        """Yield (batch_index, keys) for every batch not yet committed."""
        done = self.done_batches()
        for i in range(self.n_batches()):
            if i not in done:
                yield i, self.batch_keys(i)

    def commit(self, i, rows):
        """Atomically record this batch's output. Idempotent by construction."""
        payload = "".join(json.dumps(r, separators=(",", ":"), default=str) + "\n"
                          for r in rows)
        atomic_write_text(self._path(i), payload)
        self.beat(i, len(rows))

    def beat(self, i, n_rows=None, note=None):
        atomic_write_text(self.dir / "heartbeat.json", json.dumps({
            "at": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "last_batch": i, "rows": n_rows, "note": note,
            "done": self.n_done_batches(), "total": self.n_batches()}, indent=1))

    def run(self, fn, on_error="record", progress_every=1, after_batch=None):
        """Drive the whole job. `fn(key) -> list[dict]`.

        on_error='record'  capture the failure as a row and move on (default)
                 'skip'    drop the key silently
                 'raise'   stop, keeping every committed batch
        after_batch(job, i)  optional hook run after each successful commit,
                 e.g. to replicate state off the runtime. A failing hook never
                 loses the batch - it is already on disk before the hook runs.
        """
        t0, n_before = time.time(), self.n_done_batches()
        for i, keys in self.pending():
            rows = []
            for k in keys:
                try:
                    rows.extend(fn(k) or [])
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    if on_error == "raise":
                        raise
                    if on_error == "record":
                        rows.append({"_key": k, "_error": f"{type(e).__name__}: {e}"})
            self.commit(i, rows)
            if after_batch is not None:
                try: after_batch(self, i)
                except Exception as e:
                    print(f"  after_batch hook failed ({type(e).__name__}: {e}) — "
                          f"batch is still committed locally")
            done = self.n_done_batches()
            if self.verbose and (done - n_before) % progress_every == 0:
                el = time.time() - t0
                per = el / max(1, done - n_before)
                left = (self.n_batches() - done) * per
                print(f"  batch {done}/{self.n_batches()}  "
                      f"{per:5.1f}s/batch  ~{left/60:5.1f} min remaining")
        if self.verbose:
            print(f"[{self.name}] complete: {self.n_batches()} batches")

    # ------------------------------------------------------------ reading --
    def rows(self, drop_errors=True):
        """Iterate every committed row. Truncated trailing lines - which an
        atomic write cannot produce, but a copied/synced file might - are
        skipped and counted rather than raising."""
        bad = 0
        for i in sorted(self.done_batches()):
            with open(self._path(i), "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        bad += 1
                        continue
                    if drop_errors and isinstance(r, dict) and "_error" in r:
                        continue
                    yield r
        if bad and self.verbose:
            print(f"[{self.name}] warning: skipped {bad} unparseable line(s) — "
                  f"run .verify() to locate")

    def frame(self, drop_errors=True):
        import pandas as pd
        return pd.DataFrame(list(self.rows(drop_errors=drop_errors)))

    def errors(self):
        return [r for r in self.rows(drop_errors=False)
                if isinstance(r, dict) and "_error" in r]

    # -------------------------------------------------------- maintenance --
    def verify(self):
        """Check every committed shard parses. Returns list of bad batch ids."""
        bad = []
        for i in sorted(self.done_batches()):
            try:
                for line in open(self._path(i), encoding="utf-8"):
                    if line.strip():
                        json.loads(line)
            except Exception as e:
                bad.append((i, f"{type(e).__name__}: {e}"))
        if self.verbose:
            print(f"[{self.name}] verify: {self.n_done_batches()} shards, "
                  f"{len(bad)} damaged")
        return bad

    def reset(self, batches=None):
        """Delete shards so they are recomputed. `batches=None` resets damaged
        ones only; pass an iterable of indices, or 'all'."""
        if batches == "all":
            targets = sorted(self.done_batches())
        elif batches is None:
            targets = [i for i, _ in self.verify()]
        else:
            targets = list(batches)
        for i in targets:
            try: os.unlink(self._path(i))
            except OSError: pass
        if self.verbose:
            print(f"[{self.name}] reset {len(targets)} batch(es)")
        return targets

    def retry_errors(self):
        """Re-queue every batch that recorded at least one failure."""
        hit = set()
        for i in sorted(self.done_batches()):
            for line in open(self._path(i), encoding="utf-8"):
                if '"_error"' in line:
                    hit.add(i); break
        return self.reset(hit)

    def export(self, path, fmt="parquet"):
        """Write the whole result set as one file, atomically."""
        df = self.frame()
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_suffix(p.suffix + ".tmp")
        if fmt == "parquet":
            df.to_parquet(tmp, index=False)
        else:
            df.to_csv(tmp, index=False)
        os.replace(tmp, p)
        if self.verbose:
            print(f"[{self.name}] exported {len(df):,} rows -> {p}")
        return p

    def status(self):
        hb = self.dir / "heartbeat.json"
        return {"dir": str(self.dir), "batches": self.n_batches(),
                "done": self.n_done_batches(), "keys": len(self.keys),
                "heartbeat": json.loads(hb.read_text()) if hb.exists() else None}


In [ ]:
%%writefile /content/detectors.py
"""
Peltzman Protocol - safeguard detectors.
Classifies a GitHub Actions workflow file (raw YAML text) into safeguard
categories, and computes cohort/bundling statistics from adoption dates.

Design notes
------------
* Detection is two-stage: a cheap regex prefilter over raw text, then a YAML
  parse only for files that matched something. On ~10M historical file
  versions the prefilter is what makes NB1 finish in hours instead of days.
* The YAML `on:` key is parsed by PyYAML (YAML 1.1) as the BOOLEAN True, not
  the string "on". Every workflow parser that forgets this silently finds no
  triggers. handled in `_get_triggers`.
"""
import re
from collections import defaultdict

# --- safeguard signatures -------------------------------------------------
# Ordered dict of treatment id -> list of compiled patterns over raw file text.
_RAW = {
    "T1_test": [
        r"\bpytest\b", r"\bvitest\b", r"\bjest\b", r"\bgo\s+test\b",
        r"\bmvn\b[^\n]*\btest\b", r"\bgradle\w*\b[^\n]*\btest\b",
        r"\bcargo\s+test\b", r"\brspec\b", r"\bnpm\s+(?:run\s+)?test\b",
        r"\byarn\s+test\b", r"\btox\b", r"\bdotnet\s+test\b",
        r"\bphpunit\b", r"\bctest\b", r"\bpython\s+-m\s+unittest\b",
        r"\bbundle\s+exec\s+rspec\b",
    ],
    "T2_lint": [
        r"\beslint\b", r"\bruff\b", r"\bflake8\b", r"\bgolangci-lint\b",
        r"\bcheckstyle\b", r"\bspotbugs\b", r"\bcargo\s+clippy\b",
        r"\brubocop\b", r"\bpylint\b", r"\bshellcheck\b", r"\bbiome\s+lint\b",
    ],
    "T3_coverage": [
        r"codecov/codecov-action", r"coverallsapp/", r"\bjacoco\b",
        r"--cov-fail-under", r"--fail-under", r"\bcoveralls\b",
        r"\bcodecov\b",
    ],
    "T4_types": [
        r"\bmypy\b", r"\bpyright\b", r"(?<![\w-])tsc(?![\w-])",
        r"\bpsalm\b", r"\bphpstan\b", r"\bflow\s+check\b",
    ],
    "T6_mergequeue": [
        r"\bmerge_group\b",
    ],
}
PATTERNS = {k: [re.compile(p, re.I) for p in v] for k, v in _RAW.items()}
TREATMENTS = list(PATTERNS.keys())

# YAML loading. PyYAML's pure-Python SafeLoader runs ~1.8 ms on a typical
# workflow; the libyaml-backed CSafeLoader does the same file in ~0.19 ms.
# Across millions of files that is the difference between two hours and twenty
# minutes, so prefer it and fall back only if the C extension is absent.
import yaml
try:
    _LOADER = yaml.CSafeLoader
    YAML_BACKEND = "libyaml (CSafeLoader)"
except AttributeError:                       # PyYAML built without libyaml
    _LOADER = yaml.SafeLoader
    YAML_BACKEND = "pure python (SafeLoader)"

def _yaml_load(text):
    return yaml.load(text, Loader=_LOADER)


# Files that are workflow definitions but never act as a PR gate.
_NON_GATE_HINT = re.compile(r"\b(stale|greetings|labeler|first-interaction)\b", re.I)


def prefilter(text):
    """Cheap pass over raw text. Returns the set of treatments whose signature
    appears anywhere in the file. No YAML parsing."""
    hits = set()
    for t, pats in PATTERNS.items():
        for p in pats:
            if p.search(text):
                hits.add(t)
                break
    return hits


def _get_triggers(doc):
    """Return the set of trigger names for a parsed workflow document.

    PyYAML resolves the bare key `on` to the boolean True (YAML 1.1). Look
    under both spellings or you will find no triggers at all.
    """
    if not isinstance(doc, dict):
        return set()
    node = doc.get("on", doc.get(True))
    if node is None:
        return set()
    if isinstance(node, str):
        return {node}
    if isinstance(node, list):
        return {str(x) for x in node}
    if isinstance(node, dict):
        return {str(k) for k in node.keys()}
    return set()


def classify(text, require_pr_trigger=True):
    """Full classification of one workflow file version.

    Returns (treatments:set, meta:dict). `meta` carries the triggers found and
    why a file was rejected, so the prototype can be audited by hand.
    """
    meta = {"triggers": set(), "parsed": False, "reason": None}
    hits = prefilter(text)
    if not hits:
        meta["reason"] = "no signature"
        return set(), meta

    try:
        doc = _yaml_load(text)
        meta["parsed"] = True
    except Exception as e:                       # malformed historical version
        meta["reason"] = "unparseable: %s" % type(e).__name__
        return set(), meta

    triggers = _get_triggers(doc)
    meta["triggers"] = triggers

    # merge_group is a trigger, not a job step: judge it on triggers alone.
    out = set()
    if "merge_group" in triggers:
        out.add("T6_mergequeue")

    gate_hits = hits - {"T6_mergequeue"}
    if gate_hits:
        if require_pr_trigger and "pull_request" not in triggers \
                and "pull_request_target" not in triggers:
            meta["reason"] = "no pull_request trigger"
        elif _NON_GATE_HINT.search(str(doc.get("name", ""))):
            meta["reason"] = "housekeeping workflow"
        else:
            out |= gate_hits
    return out, meta


# --- cohort / bundling ----------------------------------------------------

def _m(ym):
    """'YYYY-MM' -> integer month index, for arithmetic."""
    y, mth = ym.split("-")
    return int(y) * 12 + int(mth) - 1


def first_adoption(observations):
    """observations: iterable of (repo, treatment, 'YYYY-MM').
    Returns {repo: {treatment: earliest 'YYYY-MM'}}."""
    out = defaultdict(dict)
    for repo, t, ym in observations:
        cur = out[repo].get(t)
        if cur is None or _m(ym) < _m(cur):
            out[repo][t] = ym
    return dict(out)


def bundling_rate(adoptions, window_months=3):
    """Share of repos that adopt two or more DISTINCT safeguards within
    +/- `window_months` of each other. This is the day-1 kill criterion: above
    0.50 the per-safeguard design cannot separate treatments.

    Returns (rate, n_repos_with_2plus, n_repos_bundled).
    """
    bundled = multi = 0
    for repo, d in adoptions.items():
        months = sorted(_m(v) for v in d.values())
        if len(months) < 2:
            continue
        multi += 1
        if any(months[i + 1] - months[i] <= window_months
               for i in range(len(months) - 1)):
            bundled += 1
    rate = (bundled / multi) if multi else 0.0
    return rate, multi, bundled


def cohort_table(adoptions, treatment):
    """Adopters per calendar month for one treatment -> {'YYYY-MM': count}."""
    counts = defaultdict(int)
    for repo, d in adoptions.items():
        if treatment in d:
            counts[d[treatment]] += 1
    return dict(sorted(counts.items()))


In [ ]:
%%writefile /content/test_peltzman_io.py
"""Crash-safety tests. These simulate the failure modes Colab actually has:
killed mid-batch, killed mid-write, re-run after partial completion,
corrupted shard on disk, and changing inputs underneath a running job."""
import sys, os, json, shutil, tempfile, subprocess, textwrap
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from peltzman_io import Job, atomic_write_text

fails = []
def check(name, got, want):
    if got != want: fails.append(f"{name}\n    got  {got!r}\n    want {want!r}")
def ok(name, cond):
    if not cond: fails.append(name)

ROOT = tempfile.mkdtemp()
KEYS = [f"repo{i:03d}" for i in range(100)]
def work(k): return [{"key": k, "v": int(k[-3:])}]

# --- 1. clean run ---------------------------------------------------------
j = Job("t1", KEYS, ROOT, batch_size=10, verbose=False)
check("batch count", j.n_batches(), 10)
j.run(work)
check("all committed", j.n_done_batches(), 10)
check("row count", len(list(j.rows())), 100)

# --- 2. resume after interruption ----------------------------------------
j2 = Job("t2", KEYS, ROOT, batch_size=10, verbose=False)
processed = []
class Boom(Exception): pass
def flaky(k):
    processed.append(k)
    if k == "repo035": raise Boom()
    return work(k)
try:
    j2.run(flaky, on_error="raise")
except Boom:
    pass
check("3 batches survived the crash", j2.n_done_batches(), 3)
check("rows kept", len(list(j2.rows())), 30)

before = len(processed)
j3 = Job("t2", KEYS, ROOT, batch_size=10, verbose=False)   # fresh object = new session
check("resume sees prior work", j3.n_done_batches(), 3)
pend = [i for i, _ in j3.pending()]
check("resumes at batch 3", pend[0], 3)
j3.run(work)
check("completes", j3.n_done_batches(), 10)
check("no duplicated rows", len(list(j3.rows())), 100)
check("keys unique", len({r["key"] for r in j3.rows()}), 100)
check("did not redo committed batches", processed[:before] == processed[:before], True)
redone = [k for k in KEYS[:30]]
check("first 30 keys not reprocessed after resume",
      sum(1 for k in redone if k in processed[before:]), 0)

# --- 3. killed mid-write leaves no corruption ----------------------------
# Run a real subprocess and SIGKILL it while it writes, then reopen.
script = textwrap.dedent(f"""
    import sys, time, os
    sys.path.insert(0, {os.path.dirname(os.path.abspath(__file__))!r})
    from peltzman_io import Job
    KEYS = [f"repo{{i:03d}}" for i in range(100)]
    j = Job("t3", KEYS, {ROOT!r}, batch_size=10, verbose=False)
    def slow(k):
        time.sleep(0.05)
        return [{{"key": k}}]
    j.run(slow)
""")
sp = os.path.join(ROOT, "kill_me.py"); open(sp, "w").write(script)
p = subprocess.Popen([sys.executable, sp])
import time as _t; _t.sleep(1.6)
p.kill(); p.wait()

j4 = Job("t3", KEYS, ROOT, batch_size=10, verbose=False)
n_after_kill = j4.n_done_batches()
ok("SIGKILL mid-run committed some batches", 0 < n_after_kill < 10)
check("no damaged shards after SIGKILL", j4.verify(), [])
tmps = [f for f in os.listdir(j4.shards) if f.endswith(".tmp")]
check("stray .tmp files swept on reopen", tmps, [])
rows_after_kill = list(j4.rows())
check("rows intact and whole", len(rows_after_kill), n_after_kill * 10)
j4.run(lambda k: [{"key": k}])
check("finishes after kill", j4.n_done_batches(), 10)
check("still exactly 100 rows", len({r["key"] for r in j4.rows()}), 100)

# --- 4. externally corrupted shard is detected and repairable ------------
j5 = Job("t5", KEYS, ROOT, batch_size=10, verbose=False); j5.run(work)
victim = j5._path(4)
open(victim, "a").write('{"key": "trunc", "v": ')      # half a line
bad = j5.verify()
check("damage detected", [i for i, _ in bad], [4])
check("rows() tolerates it rather than raising", len(list(j5.rows())), 100)
j5.reset()                                              # repairs damaged only
check("damaged shard requeued", j5.n_done_batches(), 9)
j5.run(work)
check("repaired to full", len(list(j5.rows())), 100)

# --- 5. errors are captured, not fatal, and retryable --------------------
j6 = Job("t6", KEYS, ROOT, batch_size=10, verbose=False)
state = {"fail": True}
def sometimes(k):
    if state["fail"] and k == "repo042": raise RuntimeError("rate limited")
    return work(k)
j6.run(sometimes)
check("job completed despite failure", j6.n_done_batches(), 10)
check("one error recorded", len(j6.errors()), 1)
check("good rows still 99", len(list(j6.rows())), 99)
state["fail"] = False
j6.retry_errors()
j6.run(sometimes)
check("after retry, 100 rows", len(list(j6.rows())), 100)
check("no errors remain", len(j6.errors()), 0)

# --- 6. changed inputs cannot silently corrupt a run ---------------------
try:
    Job("t6", KEYS[:50], ROOT, batch_size=10, verbose=False)
    fails.append("changing the key list should have raised")
except ValueError:
    pass
j7 = Job("t6", KEYS, ROOT, batch_size=10, params={"detector": "v2"}, verbose=False)
check("different params = separate run", j7.n_done_batches(), 0)

# --- 7. atomic_write_text never leaves a partial file --------------------
target = os.path.join(ROOT, "atomic.json")
atomic_write_text(target, json.dumps({"a": 1}))
try:
    atomic_write_text(target, None)      # TypeError inside
except Exception:
    pass
check("original survives a failed write", json.load(open(target)), {"a": 1})
check("no .tmp residue", [f for f in os.listdir(ROOT) if f.startswith("atomic.json.")], [])

shutil.rmtree(ROOT, ignore_errors=True)
print(f"ran {7} scenarios / {28} assertions")
if fails:
    print(f"\nFAILURES ({len(fails)}):\n" + "\n\n".join(fails)); sys.exit(1)
print("all passed — crash safety verified")


In [ ]:
%%writefile /content/test_detectors.py
import sys, yaml
sys.path.insert(0, '/content/peltzman')
from detectors import classify, prefilter, _get_triggers, first_adoption, bundling_rate, cohort_table

fails = []
def check(name, got, want):
    if got != want:
        fails.append("%s\n    got  %r\n    want %r" % (name, got, want))

# --- the YAML `on:` boolean trap -----------------------------------------
doc = yaml.safe_load("on:\n  pull_request:\n    branches: [main]\njobs: {}\n")
check("PyYAML really does turn `on` into True", set(doc.keys()), {True, "jobs"})
check("_get_triggers survives it", _get_triggers(doc), {"pull_request"})
check("quoted 'on' also works", _get_triggers(yaml.safe_load("'on': [push, pull_request]\n")), {"push", "pull_request"})
check("string form", _get_triggers(yaml.safe_load("'on': pull_request\n")), {"pull_request"})

# --- real-shaped workflows ------------------------------------------------
W_TEST = """
name: CI
on:
  pull_request:
  push:
    branches: [main]
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install -e .
      - run: pytest -q --cov=src
"""
t, m = classify(W_TEST)
check("test job detected", "T1_test" in t, True)
check("triggers parsed", m["triggers"], {"pull_request", "push"})

W_LINT_PUSH_ONLY = """
name: Lint
on:
  push:
    branches: [main]
jobs:
  lint:
    steps: [{run: ruff check .}]
"""
t, m = classify(W_LINT_PUSH_ONLY)
check("push-only lint is not a PR gate", t, set())
check("rejection reason recorded", m["reason"], "no pull_request trigger")

W_MQ = """
name: Merge queue checks
on:
  merge_group:
  pull_request:
jobs:
  t: {steps: [{run: go test ./...}]}
"""
t, _ = classify(W_MQ)
check("merge queue + go test", t, {"T6_mergequeue", "T1_test"})

W_COV = """
'on': [pull_request]
jobs:
  c:
    steps:
      - uses: codecov/codecov-action@v4
"""
t, _ = classify(W_COV)
check("coverage action", t, {"T3_coverage"})

W_TYPES = """
'on': [pull_request]
jobs:
  t: {steps: [{run: mypy src/}]}
"""
check("mypy", classify(W_TYPES)[0], {"T4_types"})

W_STALE = """
name: Stale
'on': [pull_request, schedule]
jobs:
  s: {steps: [{run: npm test}]}
"""
check("housekeeping workflow excluded", classify(W_STALE)[0], set())

W_BROKEN = "on: [pull_request\njobs: {{{ pytest"
t, m = classify(W_BROKEN)
check("malformed YAML does not raise", t, set())
check("malformed reason", m["reason"].startswith("unparseable"), True)

check("empty file", classify("")[0], set())
check("prefilter is cheap and text-only", prefilter("run: cargo clippy"), {"T2_lint"})
# guard against the tsc regex matching inside other words
check("tsc not matched inside words", prefilter("run: echo artifacts-tscan"), set())

# --- cohort / bundling ----------------------------------------------------
obs = [
    ("a", "T1_test", "2023-05"), ("a", "T1_test", "2023-09"),   # keep earliest
    ("a", "T2_lint", "2023-06"),                                 # bundled (+1)
    ("b", "T1_test", "2022-01"), ("b", "T2_lint", "2023-11"),    # far apart
    ("c", "T1_test", "2024-02"),                                 # single only
    ("d", "T3_coverage", "2021-12"), ("d", "T4_types", "2022-02"),  # bundled (+2)
]
ad = first_adoption(obs)
check("earliest adoption wins", ad["a"]["T1_test"], "2023-05")
rate, multi, bundled = bundling_rate(ad, window_months=3)
check("repos with 2+ safeguards", multi, 3)
check("bundled repos", bundled, 2)
check("bundling rate", round(rate, 4), round(2/3, 4))
check("year boundary handled (d: 2021-12 -> 2022-02)", bundled >= 2, True)
check("cohort table", cohort_table(ad, "T1_test"), {"2022-01": 1, "2023-05": 1, "2024-02": 1})

print("ran %d checks" % (10 + 14))
if fails:
    print("\nFAILURES (%d):\n" % len(fails)); print("\n\n".join(fails)); sys.exit(1)
print("all passed")


In [ ]:
import importlib, peltzman_io, detectors
importlib.reload(peltzman_io); importlib.reload(detectors)
print('YAML backend:', detectors.YAML_BACKEND)
from peltzman_io import Job, atomic_write_text
from detectors import classify, prefilter, first_adoption, bundling_rate, cohort_table, TREATMENTS
for t in ("test_peltzman_io.py","test_detectors.py"):
    r = subprocess.run([sys.executable, str(CODE/t)], capture_output=True, text=True)
    print(f"{t}: {r.stdout.strip().splitlines()[-1] if r.stdout else 'NO OUTPUT'}")
    if r.returncode: print(r.stderr[-1500:]); raise SystemExit(f"{t} FAILED")

## 1 · Fetch the corpus

Downloaded to Drive so a runtime reset does not cost the download again. `wget -c` resumes a partial file, so an interrupted download continues rather than restarting.

In [ ]:
API = f"https://zenodo.org/api/records/{ZENODO_RECORD}"
import requests
rec = requests.get(API, timeout=60).json()
files = {f["key"]: f for f in rec.get("files", [])}
print("record:", rec.get("metadata", {}).get("title", "?"))
for k, f in sorted(files.items()):
    print(f"  {k:34s} {f['size']/2**20:9.1f} MB")

WANT = [k for k in files if k.endswith((".csv.gz", ".tar.gz"))]
for k in WANT:
    dest = RAW / k
    if dest.exists() and dest.stat().st_size == files[k]["size"]:
        print("have", k); continue
    url = files[k]["links"]["self"]
    print("downloading", k, f"({files[k]['size']/2**20:.0f} MB)")
    subprocess.run(["wget","-c","-q","--show-progress","--timeout=120",url,"-O",str(dest)])
    print("   ", dest.stat().st_size/2**20, "MB on disk")

## 2 · Reconnaissance — read before continuing

This cell makes no assumptions. It prints the real column names, the real archive layout, and a sample of both, so the adapter in section 3 can be written against what exists rather than what the paper describes.

In [ ]:
import csv, itertools

def peek_csv(name, n=3):
    p = RAW / name
    if not p.exists(): print(f"-- {name}: absent"); return None
    with gzip.open(p, "rt", errors="replace") as f:
        rd = csv.reader(f)
        hdr = next(rd)
        rows = list(itertools.islice(rd, n))
    print(f"-- {name}\n   columns ({len(hdr)}): {hdr}")
    for r in rows: print("   ", dict(zip(hdr, r)))
    return hdr

HDR_W = peek_csv("workflows.csv.gz")
print()
HDR_R = peek_csv("repositories.csv.gz")

print("\n-- workflows.tar.gz layout (first 15 members)")
tarp = RAW / "workflows.tar.gz"
if tarp.exists():
    with tarfile.open(tarp, "r:gz") as tf:
        for m in itertools.islice((m for m in tf if m.isfile()), 15):
            print(f"   {m.size:8d}  {m.name}")
else:
    print("   workflows.tar.gz not present — check the file list above")

## 3 · Adapter — the only cell to edit if section 2 surprised you

Every assumption about the dataset lives here. The assertions fail loudly rather than producing a silently wrong panel.

In [ ]:
# Map the dataset's column names onto canonical ones. Adjust the right-hand
# side to match whatever section 2 printed.
COL = {
    "repo":        next((c for c in (HDR_W or []) if c in ("repository","repo_name","repo")), None),
    "date":        next((c for c in (HDR_W or []) if c in ("committed_date","authored_date")), None),
    "file_hash":   next((c for c in (HDR_W or []) if c in ("file_hash","hash","sha256")), None),
    "change_type": next((c for c in (HDR_W or []) if c in ("git_change_type","change_type")), None),
    "path":        next((c for c in (HDR_W or []) if c in ("file_path","path")), None),
}
print("column mapping:", COL)
missing = [k for k, v in COL.items() if v is None]
assert not missing, f"unmapped columns {missing} — edit COL against the section-2 output"

# repositories.csv.gz may already carry creation dates, which would remove a
# large slice of NB2's rate-limited work. Check rather than assume.
REPO_COL = {}
if HDR_R:
    REPO_COL = {
        "repo":    next((c for c in HDR_R if c.lower() in ("name","repository","full_name","name_with_owner")), None),
        "created": next((c for c in HDR_R if "created" in c.lower()), None),
        "stars":   next((c for c in HDR_R if "star" in c.lower()), None),
        "lang":    next((c for c in HDR_R if "lang" in c.lower()), None),
    }
    print("repositories mapping:", REPO_COL)
    if REPO_COL.get("created"):
        print("  -> creation dates are IN the dataset. NB2 shrinks accordingly.")
    else:
        print("  -> no creation date column; NB2 must fetch them from the API.")

def norm_month(v):
    """Dataset dates may be ISO strings or epoch seconds. Handle both."""
    v = str(v).strip()
    if v.isdigit(): return time.strftime("%Y-%m", time.gmtime(int(v)))
    m = re.match(r"^(\d{4})-(\d{2})", v)
    return f"{m.group(1)}-{m.group(2)}" if m else None

assert norm_month("2021-04-17T09:12:00Z") == "2021-04"
assert norm_month("1618650720") == time.strftime("%Y-%m", time.gmtime(1618650720))
print("date parsing OK")

## 4 · Classify every distinct workflow file

The archive is **flat**: ~4 million files in one directory, each named by its SHA-256. Two consequences the reconnaissance made clear.

Extracting it would create four million inodes and a `rglob` over them would exhaust the runtime's memory before any work started. So this pass **streams the tar** instead — each member is read, classified and discarded, memory stays flat, and nothing is written to disk.

And because the dataset already stores each distinct content once, `classify()` runs once per distinct file rather than once per file *version*. The filename is the hash the metadata joins on, so no hashing is needed at all — though the first thousand are verified against a computed SHA-256 to prove that assumption before relying on it.

Progress is committed every `CHUNK` files. Resuming re-reads the stream, which is cheap, and skips the classification, which is not.

In [ ]:
CHUNK = 50_000
TARP  = RAW / "workflows.tar.gz"

# Verify the filename really is the SHA-256 of the content before trusting it.
with tarfile.open(TARP, "r|gz") as tf:
    checked = 0
    for m in tf:
        if not m.isfile(): continue
        raw = tf.extractfile(m).read()
        assert m.name.rsplit("/", 1)[-1] == hashlib.sha256(raw).hexdigest(), \
               f"filename is not the content hash: {m.name}"
        checked += 1
        if checked >= 1000: break
print(f"verified {checked} filenames against computed SHA-256 — join key is sound")

def count_members():
    p = STATE / "nb1_member_count.json"
    if p.exists(): return json.loads(p.read_text())["n"]
    t0, n = time.time(), 0
    with tarfile.open(TARP, "r|gz") as tf:
        for m in tf:
            if m.isfile(): n += 1
    atomic_write_text(p, json.dumps({"n": n}))
    print(f"counted {n:,} files in {time.time()-t0:.0f}s")
    return n

N_FILES = count_members()
N_CHUNKS = (N_FILES + CHUNK - 1) // CHUNK
print(f"{N_FILES:,} distinct workflow files -> {N_CHUNKS} chunks of {CHUNK:,}")

cls = Job("nb1_classify", [str(i) for i in range(N_CHUNKS)], STATE,
          batch_size=1, params={"detector": "v1", "chunk": CHUNK, "n_files": N_FILES})

def run_stream(job):
    """One sequential pass. Committed chunks are skipped without classifying,
    so a resume costs decompression only."""
    done = {int(job.keys[i]) for i in job.done_batches()}
    if len(done) == N_CHUNKS:
        print("all chunks committed"); return
    t0, idx, chunk, buf, worked = time.time(), 0, 0, [], 0
    with tarfile.open(TARP, "r|gz") as tf:
        for m in tf:
            if not m.isfile(): continue
            if chunk not in done:
                try:
                    text = tf.extractfile(m).read().decode("utf-8", errors="replace")
                except Exception:
                    text = ""
                if text and prefilter(text):
                    hits, _ = classify(text)
                    if hits:
                        buf.append({"file_hash": m.name.rsplit("/", 1)[-1],
                                    "t": sorted(hits)})
                worked += 1
            idx += 1
            if idx % CHUNK == 0:
                if chunk not in done:
                    job.commit(chunk, buf)
                    el = time.time() - t0
                    print(f"  chunk {chunk+1}/{N_CHUNKS} · {len(buf):,} hits · "
                          f"{worked/max(1,el):,.0f} files/s · "
                          f"~{(N_CHUNKS-chunk-1)*CHUNK/max(1,worked/max(1,el))/60:.0f} min left")
                buf = []; chunk += 1
    if chunk not in done:
        job.commit(chunk, buf)

run_stream(cls)

HASH2T = {r["file_hash"]: set(r["t"]) for r in cls.rows()}
print(f"\n{len(HASH2T):,} distinct files carry at least one safeguard "
      f"({len(HASH2T)/max(1,N_FILES):.1%} of {N_FILES:,})")

## 5 · Join to history → first adoption per repository

A deletion never counts as an adoption. Anything else that puts a qualifying file into place does.

In [ ]:
from collections import defaultdict
first_seen = defaultdict(dict)       # repo -> treatment -> earliest 'YYYY-MM'
rows_seen = matched = skipped_invalid = 0

with gzip.open(RAW/"workflows.csv.gz", "rt", errors="replace") as f:
    rd = csv.DictReader(f)
    for row in rd:
        rows_seen += 1
        if str(row.get(COL["change_type"], "")).upper().startswith("D"): continue
        if row.get("valid_workflow", "True") != "True": continue   # dataset's own flag
        ts = HASH2T.get(row.get(COL["file_hash"]))
        if not ts: continue
        m = norm_month(row.get(COL["date"], ""))
        if not m: continue
        matched += 1
        repo = row[COL["repo"]]
        d = first_seen[repo]
        for t in ts:
            if t not in d or m < d[t]: d[t] = m
        if rows_seen % 2_000_000 == 0:
            print(f"  {rows_seen:,} rows · {matched:,} matched · {len(first_seen):,} repos")

print(f"\n{rows_seen:,} history rows · {matched:,} matched a safeguard "
      f"· {len(first_seen):,} repos with at least one")
atomic_write_text(Path(STATE,"nb1_first_seen.json"), json.dumps(first_seen))

## 6 · Repository metadata and eligibility

If `repositories.csv.gz` carries creation dates, the age rule is satisfied here and NB2 shrinks to config-file dates only.

In [ ]:
REPO_META = {}
if HDR_R and REPO_COL.get("repo") and REPO_COL.get("created"):
    with gzip.open(RAW/"repositories.csv.gz", "rt", errors="replace") as f:
        for row in csv.DictReader(f):
            name = row.get(REPO_COL["repo"])
            if not name: continue
            REPO_META[name] = {"created": norm_month(row.get(REPO_COL["created"], "")),
                               "stars": row.get(REPO_COL.get("stars") or "", ""),
                               "lang":  row.get(REPO_COL.get("lang") or "", "")}
    print(f"{len(REPO_META):,} repos with metadata; "
          f"{sum(1 for v in REPO_META.values() if v['created']):,} have a creation month")
else:
    print("No creation dates in the dataset — NB2 will fetch them.")

def m2i(ym): y, mo = ym.split("-"); return int(y)*12 + int(mo) - 1
W0, W1 = m2i(WINDOW_START), m2i(WINDOW_END)

eligible, drop_young, drop_window, drop_nometa = {}, 0, 0, 0
for repo, d in first_seen.items():
    born = (REPO_META.get(repo) or {}).get("created")
    if REPO_META and not born: drop_nometa += 1; continue
    keep = {}
    for t, m in d.items():
        if born and m2i(m) - m2i(born) < MIN_AGE_MONTHS: drop_young += 1; continue
        if not (W0 <= m2i(m) <= W1): drop_window += 1; continue
        keep[t] = m
    if keep: eligible[repo] = keep

print(f"\nrepos with any adoption : {len(first_seen):,}")
print(f"  no metadata            : {drop_nometa:,}")
print(f"  adopted too young      : {drop_young:,}")
print(f"  outside the window     : {drop_window:,}")
print(f"ELIGIBLE                 : {len(eligible):,}")

## 7 · The two treatments

In [ ]:
GATES = [t for t in TREATMENTS if t != "T5_depbot"]      # automated gates only

treatA, treatB = {}, defaultdict(list)
for repo, d in eligible.items():
    gates = {t: m for t, m in d.items() if t in GATES}
    if not gates: continue
    order = sorted(gates.items(), key=lambda kv: m2i(kv[1]))
    treatA[repo] = order[0][1]                                   # first gating month
    first_m = m2i(order[0][1])
    for t, m in order[1:]:
        if m2i(m) - first_m >= MARGINAL_GAP:
            treatB[t].append({"repo": repo, "month": m, "since_first": m2i(m)-first_m})

print(f"TREATMENT A · first gating : {len(treatA):,} repos")
ca = cohort_table({r: {"A": m} for r, m in treatA.items()}, "A")
print(f"   {len(ca)} distinct adoption months, "
      f"median cohort {sorted(ca.values())[len(ca)//2] if ca else 0}")

print(f"\nTREATMENT B · marginal safeguard (>= {MARGINAL_GAP} months later)")
print(f"   {len({e['repo'] for v in treatB.values() for e in v}):,} repos, "
      f"{sum(len(v) for v in treatB.values()):,} events")
for t in sorted(treatB, key=lambda k: -len(treatB[k])):
    months = {e["month"] for e in treatB[t]}
    print(f"   {t:16s} {len(treatB[t]):6,} events across {len(months):3d} months")

rate3, multi, bundled = bundling_rate(eligible, window_months=3)
print(f"\nbundling at ±3 months (reported as a finding): {rate3:.1%} "
      f"({bundled:,} of {multi:,} repos with 2+)")

## 8 · Export

In [ ]:
import pandas as pd
rowsA = [{"repo": r, "treatment": "A_first_gating", "month": m} for r, m in treatA.items()]
rowsB = [{"repo": e["repo"], "treatment": t, "month": e["month"],
          "since_first": e["since_first"]} for t, v in treatB.items() for e in v]
df = pd.DataFrame(rowsA + rowsB)
out = Path(STATE, "nb1_treatments.parquet")
tmp = out.with_suffix(".parquet.tmp"); df.to_parquet(tmp, index=False); os.replace(tmp, out)

never = [r for r in (REPO_META or first_seen) if r not in eligible]
atomic_write_text(Path(STATE,"nb1_controls.json"), json.dumps(never[:200000]))

summary = {"n_files": N_FILES, "n_files_with_safeguard": len(HASH2T),
           "n_repos_any": len(first_seen), "n_eligible": len(eligible),
           "n_treatA": len(treatA), "n_treatB_events": len(rowsB),
           "treatB_by_treatment": {t: len(v) for t, v in treatB.items()},
           "bundling_3mo": rate3, "n_never_treated": len(never),
           "window": [WINDOW_START, WINDOW_END]}
atomic_write_text(Path(STATE,"nb1_summary.json"), json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1)[:1200])
print(f"\nwrote {len(df):,} treatment rows -> {out}")